In [1]:
import os
print(os.environ.get("SYMBOLICA_LICENSE"))

None


In [2]:
import os
os.environ["SYMBOLICA_LICENSE"] = "3bcddd98#6b8ff38b#27278ef0-ec70-5376-96a7-7dca1939dc28"

In [9]:
from symbolica import *
from IPython.display import Markdown as md, display as dp
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

def display(self) -> None:
    dp(md(self.to_latex().replace('gamma', '\\gamma').replace('mu', '\\mu')))

def display_graph(self: Graph) -> None:
    dp(md('```mermaid\n' + self.to_mermaid() + '\n```'))

## Canonize Tensor

Canonize (products of) tensors in the expression by relabeling repeated indices. The tensors must be written as functions, with its indices as the arguments. Subexpressions, constants and open indices are supported.

If the contracted indices are distinguishable (for example in their dimension), you can provide a group marker as the second element in the tuple of the index specification. This makes sure that an index will not be renamed to an index from a different group.

In [17]:


phi = S("phi")
i, j, k, d1, d2 = S("i", "j", "k", "d1", "d2")

e1 = phi(i) * phi(i)
e2 = phi(j) * phi(j)
e3 = phi(k) * phi(k)
display(e2)
display(e3)
print("e1 =", e1)
print("e2 =", e2)
print("e3 =", e3)

print("canon(e1) =", e1.canonize_tensors([(i, 0), (j, 0), (k, 0), (d1, 0), (d2, 0)]))
print("canon(e2) =", e2.canonize_tensors([(i, 0), (j, 0), (k, 0), (d1, 0), (d2, 0)]))
print("canon(e3) =", e3.canonize_tensors([(i, 0), (j, 0), (k, 0), (d1, 0), (d2, 0)]))

$$phi\!\left(j\right)^{2}$$

$$phi\!\left(k\right)^{2}$$

e1 = phi(i)^2
e2 = phi(j)^2
e3 = phi(k)^2
canon(e1) = (phi(i)^2, [], [(i, 0)])
canon(e2) = (phi(i)^2, [], [(i, 0)])
canon(e3) = (phi(i)^2, [], [(i, 0)])


In [18]:


phi = S("phi")
i, j, a, b = S("i", "j", "a", "b")

e4 = phi(i) * phi(i) * phi(j) * phi(j)
e5 = phi(a) * phi(a) * phi(b) * phi(b)

c4 = e4.canonize_tensors([(i,0),(j,0),(a,0),(b,0)])[0]
c5 = e5.canonize_tensors([(i,0),(j,0),(a,0),(b,0)])[0]

print("e4 =", e4)
print("e5 =", e5)
print("canon(e4) =", c4)
print("canon(e5) =", c5)
print(c4.to_canonical_string() == c5.to_canonical_string())

e4 = phi(i)^2*phi(j)^2
e5 = phi(a)^2*phi(b)^2
canon(e4) = phi(i)^2*phi(j)^2
canon(e5) = phi(i)^2*phi(j)^2
True


## Pattern matching

In [11]:
x, f = S('x', 'f')
e = x + f(x) + 5
display(e)
e = e.replace(x, 6)
display(e)
print(e)

$$5+x+f\!\left(x\right)$$

$$11+f\!\left(6\right)$$

11+f(6)


In [12]:
x, y, f = S('x', 'y', 'f')
k = f(x, y).replace_multiple([Replacement(x, y), Replacement(y, x)])
print(k)

f(y,x)


### Wild cards

In [15]:
n_, f = S('n_', 'f')
e = f(4)
e = e.replace(f(0), 1)
e = e.replace(f(n_), n_*f(n_-1))
print(e)

4*f(3)


In [21]:

x___, f, x, y,z,x_ = S('x___', 'f', 'x', 'y','z','x_')

e = x*y*f(x*y)
e = e.replace(x___*f(x___), 1)

print(e)

1


In [23]:
e = x*y*z
e = e.replace(x_, 1)
print(e)

1


### Ordering


A wildcard will match to a given set of atoms only once and it will have the order of the normal form. This means that for a symmetric function f(1,2,3), f(x___) will generate one match, x___ = 1,2,3, and it will never permute through all options (e.g. x___ = 2,3,1). Consequently,

In [28]:
f = S('f')
f_sym = S('fsym', is_symmetric=True)
e = f(3,2,1)*f_sym(1,2,3)
e = e.replace(f(x___)*f_sym(x___), f(1))
print(e)

f(3,2,1)*fsym(1,2,3)


## Condition


Wildcards can be restricted by any Condition. A Condition is an object that yields True, False, or Inconclusive when evaluated.

There are several functions on expressions that yield conditions, such as contains or matches:

In [30]:
g, a, x_ = S('g', 'a', 'x_')
g(a).replace(x_, 1, x_.contains(E('a'))) # yields 1 as a is contained in g(a)
g(2).replace(x_, 1, x_.contains(E('a'))) # no match
print(g(a).replace(x_, 1, x_.contains(E('a'))))
print(g(2).replace(x_, 1, x_.contains(E('a'))))

1
g(2)


In [35]:
g, x, x_, x__,x___ = S('g', 'x', 'x_','x__','x___')
e = g(10).replace(g(x_), g(x_-1) + g(x_-2), x_ > 1, repeat=True)
print(e)

4*g(0)+5*g(1)+4*(g(0)+2*g(1))+2*(2*g(0)+3*g(1))+2*(g(0)+3*g(1)+2*(g(0)+g(1)))+2*(2*g(0)+5*g(1)+2*(g(0)+g(1))+2*(2*g(0)+3*g(1)))


The number of underscores at the end of the wildcard name determines pre-set length restrictions. We have:

- x_: must match a single atom
- x__: must match one or more atoms
- x___: must match zero or more atoms

The length of x__ and x___ can be further restricted. For example, let us restrict the wildcard x__ to a length between 2 and 3

In [37]:
e1 = f(1).replace(f(x__), 1, x__.req_len(2, 3))
e2 = f(1,2).replace(f(x__), 1, x__.req_len(2, 3))
e3 = f(1,2,3).replace(f(x__), 1, x__.req_len(2, 3))
print(e1)
print(e2)
print(e3)

f(1)
1
1


If the wildcard has attributes / tags, the matched atoms must also have the same attributes or more.
A wildcard without attributes can still match a symmetric function.

In [ ]:
f = S('f')
fs, fs_ = S('fs', 'fs_', is_symmetric=True)

f(1).replace(fs_(1), 1)  # no match
fs(1).replace(fs_(1), 1)  # match


1

In [42]:
x = S('x')
xr, xr__ = S('xr', 'xr__', is_real=True)

f(1, 2, x, 3).replace(f(xr__), 1)  # no match
f(1, 2, xr**2 + 2, 3).replace(f(xr__), 1)  # match
print(f(1, 2, x, 3).replace((xr__), 1))
print(f(1, 2, xr**2 + 2, 3).replace(xr__, 1))

f(1,1,x,1)
f(1,1,1,1)


### filter

In [44]:
from symbolica import *
x_, f = S('x_', 'f')
e = f(1)*f(2)*f(3)
e = e.replace(f(x_), 1, x_.req(lambda m: m == 2 or m == 3))
print(e)

f(1)


In [45]:
from symbolica import *
x_, y_, f = S('x_', 'y_', 'f')
e = f(1)*f(2)*f(3)
e = e.replace(f(x_)*f(y_), 1, x_.req_cmp(y_, lambda m1, m2: m1 + m2 == 4))
print(e)

f(2)


### RANGE

In [46]:
x, f = S('x', 'f')
r = x+x**2*f(x**2, x, f(x))
print(r.replace(x, 1, level_range=(1, 1)))
print(r.replace(x, 1, level_range=(2, 2)))

x+x^2*f(1,1,f(x))
x+x^2*f(x^2,x,f(1))


### Match Iteration

In [47]:
from symbolica import *
x_, f = S('x_', 'f')
e = f(1)+f(2)+f(3)

for m in e.match(f(x_)):
    print(m[x_])

1
2
3


In [48]:
from symbolica import *
x_, f = S('x_', 'f')
e = f(f(1))

for m in e.match(f(x_)):
    print(m[x_])

f(1)
1


### Replacement

In [49]:
from symbolica import *
x_, f = S('x_', 'f')
e = f(1)*f(2)*f(3)
for r in e.replace_iter(f(x_), f(x_ + 1)):
    print(r)

f(2)^2*f(3)
f(1)*f(3)^2
f(1)*f(2)*f(4)


### Delayed execution

In [50]:
from symbolica import *
x, x_, f = S('x', 'x_', 'f')
e = f((x+1)**2)
print(e.replace(f(x_), f(x_.hold(T().expand()))))

f(1+2*x+x^2)


### Matching on antisymmetric function


In [53]:
x_, y_, f = S('x_', 'y_', 'f')
fa = S('fa', is_antisymmetric=True)
r= (f(2)*f(1, 2)).replace(f(x_, y_)*f(x_), f(x_, y_)) #no match
print(r)
r = (fa(1, 2)*f(2)).replace(fa(x_, y_)*f(x_), f(x_, y_))
print(r)

f(2)*f(1,2)
f(2,1)


In [54]:
x_, y_, f = S('x_', 'y_', 'f')
fa, sign = S('fa', 'sign', is_antisymmetric=True)
r = (fa(1, 2)*f(2)).replace(fa(x_, y_)*f(x_),
                            f(x_, y_)*sign(x_, y_)).replace(sign(x_, y_), 1)
print(r)

-f(2,1)


### Coeff Extraction

In [56]:
from symbolica import *

x, y, z, n_, x_ = S('x', 'y', 'z', 'n_', 'x_')
e = x**2*(y+z) + x**3*(y+z**2)
print(e)
# match returns a dictionary that maps every wildcard to its value
coeff_list = [(m[n_], m[x_]) for m in e.match(x**n_*x_)]
for (pow, content) in coeff_list:
    print(x**pow, content)

x^2*(y+z)+x^3*(y+z^2)
x^2 y+z
x^3 y+z^2
